# Tuning de estratégias de palpite

Este caderno compara estratégias de geração de **palpites** para a Mega-Sena a partir do histórico real
de **concursos** (`data/raw/Mega-Sena.csv`, 3069 concursos), usando os módulos testados em `src/models`.
Comparamos três estratégias estatísticas (**frequência**, **atraso** e **combinada**) contra o
baseline **aleatório**, com métricas descritivas sobre o passado.

**Aviso importante:** loteria é aleatória e cada concurso é independente. As métricas abaixo descrevem
o que já saiu — **não prometem acerto, não preveem o próximo concurso e não recomendam aposta**.
O baseline aleatório existe para calibrar a expectativa: qualquer estratégia deve ser lida lado a lado com o puro acaso.

In [ ]:
# Setup: resolve repo paths and load the real concursos.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "config.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import config
from src.data import load, process
from src.models import evaluate, generate, rank, save

raw_candidates = [Path("data/raw/Mega-Sena.csv"), Path("../data/raw/Mega-Sena.csv")]
raw_path = next(p for p in raw_candidates if p.exists())
raw = load.load_raw_csv(raw_path)
df = process.process_concursos(raw)
print(f"concursos: {len(df)} | N={config.NUM_GUESSES} | seed={config.RANDOM_SEED}")
df.head(3)

## Estratégias em comparação

- **frequência:** ordena as dezenas pela contagem de aparições no histórico.
- **atraso:** ordena pela quantidade de concursos desde a última aparição.
- **combinada:** soma as duas ordens normalizadas (frequência/max + atraso/max) — equilibra as duas fontes.
- **aleatoria:** baseline uniforme, sem olhar o histórico (comparação honesta).

A geração usa a mesma semente (`config.RANDOM_SEED`) para todas: mesmo `seed`, mesmos palpites.

In [ ]:
# Ranking: top 5 dezenas por estrategia (modulos testados).
for strategy in ("frequencia", "atraso", "combinada"):
    top = rank.rank_dezenas(df, strategy=strategy).head(5)
    print(f"--- {strategy} ---")
    print(top.to_string(index=False))
    print()

### O que o ranking mostra

No fechamento do histórico, a frequência coroa as dezenas 10, 53 e 5 (mais de 330 aparições) e o atraso
destaca 34, 45 e 32 (cerca de 40 concursos sem sair). A combinada mistura as duas listas — por isso o topo
traz dezenas que são ao mesmo tempo frequentes e atrasadas (como 34 e 32). É um retrato do passado,
não um sinal de quem sai no próximo concurso.

## Palpites gerados lado a lado

Geramos `N = config.NUM_GUESSES` jogos por estratégia com a mesma semente. Cada jogo tem 6 dezenas
distintas de 1 a 60, ordenadas. A reprodutibilidade vem do `seed`: rodar de novo com o mesmo valor
repete os mesmos jogos.

In [ ]:
# Generation: N palpites per strategy with the configured seed.
strategies = ("frequencia", "atraso", "combinada", "aleatoria")
frames = {}
for strategy in strategies:
    if strategy == "aleatoria":
        frames[strategy] = generate.generate_random_palmites(
            n=config.NUM_GUESSES, seed=config.RANDOM_SEED
        )
    else:
        frames[strategy] = generate.generate_ranked_palmites(
            df, n=config.NUM_GUESSES, seed=config.RANDOM_SEED, strategy=strategy
        )
    print(f"--- {strategy} ---")
    print(frames[strategy].to_string(index=False))
    print()

## Métricas sobre o histórico

Avaliamos cada conjunto de palpites contra todos os concursos já sorteados:

- **cobertura:** dezenas distintas usadas / 60.
- **soma_media** e **pares_medio:** forma típica do jogo gerado (para comparar com o histórico: soma média ~183, pico em 3 pares).
- **acertos_medio:** média de acertos por par (palpite, concurso) no passado.
- **acertos_max:** melhor encontro com um concurso do passado.

São medidas descritivas do passado, não estimativas de acerto futuro.

In [ ]:
# Evaluation: descriptive metrics per strategy over the full history.
comparison = evaluate.compare_strategies(
    df, n=config.NUM_GUESSES, seed=config.RANDOM_SEED, strategies=strategies
)
print(comparison.to_string(index=False))

fig, (left_ax, right_ax) = plt.subplots(1, 2, figsize=(12, 4.5))
left_ax.bar(comparison["strategy"], comparison["cobertura"], color="steelblue")
left_ax.set_title("Cobertura por estratégia (distintas / 60)")
left_ax.set_xlabel("Estratégia")
left_ax.set_ylabel("Cobertura")
left_ax.tick_params(axis="x", rotation=15)
right_ax.bar(comparison["strategy"], comparison["acertos_medio"], color="darkorange")
right_ax.set_title("Acertos médios no passado por estratégia")
right_ax.set_xlabel("Estratégia")
right_ax.set_ylabel("Acertos médios (passado)")
right_ax.tick_params(axis="x", rotation=15)
fig.tight_layout()

### Justificativa da escolha

Na rodada com `seed=42` e `N=5`, os acertos médios no passado ficam em empate técnico
(todas as estratégias entre 0,59 e 0,61, com máximo 4 em 6) — diferença de ruído, não de mérito.
Nenhuma estratégia se destaca prevendo o passado, e nenhuma pode prometer o futuro.

Escolhemos a **combinada como padrão** por três motivos descritivos, não preditivos:

1. **Equilíbrio:** é a única que leva frequência e atraso juntos, sem depender de uma fonte só.
2. **Cobertura:** usa mais dezenas distintas no conjunto (maior variedade de sugestões).
3. **Forma típica:** soma e pares ficam mais próximos do centro histórico (~183, 3 pares) do que o baseline desta semente.

O baseline **aleatório segue publicado lado a lado** em `palpites.csv` para comparação honesta:
se a combinada não supera o acaso no passado, não há motivo para tratá-la como melhor no futuro.

In [ ]:
# Output: N combinada + N aleatoria side-by-side, saved without index.
ranked = generate.generate_ranked_palmites(
    df, n=config.NUM_GUESSES, seed=config.RANDOM_SEED, strategy="combinada"
)
baseline = generate.generate_random_palmites(
    n=config.NUM_GUESSES, seed=config.RANDOM_SEED
)
ranked = ranked.copy()
baseline = baseline.copy()
ranked["strategy"] = "combinada"
baseline["strategy"] = "aleatoria"
palpites = pd.concat([ranked, baseline], ignore_index=True)
out_candidates = [
    Path("data/outputs/palpites.csv"),
    Path("../data/outputs/palpites.csv"),
]
out_path = out_candidates[0] if Path.cwd().name != "notebooks" else out_candidates[1]
saved = save.save_palmites(palpites, out_path)
print(f"saved {len(palpites)} palpites -> {saved}")
print(palpites.to_string(index=False))

## Resumo

- Quatro estratégias geradas com a mesma semente (`config.RANDOM_SEED`), `N = config.NUM_GUESSES` jogos cada.
- Métricas descritivas sobre 3069 concursos: empate técnico nos acertos passados, combinada com maior cobertura.
- Escolha padrão: **combinada**, por equilíbrio entre frequência e atraso — sem promessa de acerto.
- Saída: `data/outputs/palpites.csv` com os jogos combinada + aleatória lado a lado e a coluna `strategy`, sem índice.

Loteria é aleatória. Este caderno é educacional: compara sugestões com números honestos, mas **não prevê
resultado e não recomenda aposta**.